<a href="https://colab.research.google.com/github/duyb2207513/Hugging_Face_Models_Mastery_RandD_Guideline/blob/main/Step4_Core_4_Step_Training_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Step 4: Core 4-Step Training Loop**

#**Research/Learn:**

## Deep understanding of the four fundamental steps:
  * Forward pass: the process of passing input data through the network layers to produce predictions/outputs.
  
  * Loss calculation: quantifying how wrong the model's predictions are compared to the ground truth.
  
  * Backward pass:  computing gradients of the loss with respect to every parameter using the chain rule.
  
  * Optimization: using computed gradients to adjust weights in a direction that reduces loss
   
  ## How they aplly across model types:
  
  In training loop per batch, input datas go into Forward pass( prediction)-> loss calculation(scalar error value)-> backward pass (gradients for all params) -> optimize(updated weight) and repeat for next batch

## Use of accelerate for scaling:
Accelerate is a library by Hugging Face that makes distributed training simple by abstracting away hardware-specific code. With key Features:
  * Automatic Device Placement: handles all device placement
  * Mixed Precision Training: enable automatic mixed precision (FP16)
  * Accumulate gradients over multiple steps
  * Saving model


# Actions: Implement and compare high-level Trainer versus manual training loop.

<!-- I will conduct compare Trainer API versus  -->


# Step 4: Core 4-Step Training Loop

## Objective

In Step 3, the model was trained using the high-level Trainer API.

The Trainer API automatically handles the internal deep learning training process, including:

- Forward Pass
- Loss Calculation
- Backward Pass
- Optimization

In this notebook, we manually re-implement the same training workflow using a custom PyTorch training loop in order to better understand the internal mechanics of deep learning training.

#Action 2: Implement manual training loop for model

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [3]:
import torchvision.transforms as transforms

# =========================
# 2. TRANSFORM
# =========================
transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        (0.5,),
        (0.5,)
    )
])

In [8]:
import torchvision.datasets as datasets
# =========================
# 3. LOAD MNIST
# =========================
train_data = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.07MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.30MB/s]


In [9]:

# =========================================================
# DEVICE
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda


In [12]:

# =========================================================
# DATALOADER
# =========================================================

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)
print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 938
Test batches: 157


# CNN Model

The same CNN architecture from Step 3 is reused in this notebook to ensure a fair comparison between:

1. High-level Trainer API
2. Manual 4-step training loop

In [18]:
# =========================
# 5. BUILD CNN MODEL
# =========================
class CNNModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1 ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7,128),
            nn.ReLU(),
            nn.Linear(128,10  )
        )

    def forward(
        self,
        pixel_values=None,
        labels=None
    ):

        x = self.features(pixel_values)
        logits = self.classifier(x)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(
                logits,
                labels
            )
        return {
            "loss": loss,
            "logits": logits
        }

model = CNNModel()
model.to(device)

CNNModel(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [19]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# Manual 4-Step Training Loop

Unlike the Trainer API, the following implementation manually controls every training operation.

Each iteration explicitly performs the four fundamental deep learning training steps:

1. Forward Pass
2. Loss Calculation
3. Backward Pass
4. Optimization

In [24]:
# =========================================================
# CORE 4-STEP TRAINING LOOP
# =========================================================

num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0 # Initialize for training accuracy
    total = 0   # Initialize for training accuracy

    print(f"\n========== EPOCH {epoch+1} ==========")

    for batch_idx, batch in enumerate(train_loader):

        images = batch[0].to(device)

        labels = batch[1].to(device)


        # Foward-Pass: Input goes through neural network
        outputs = model(images)


        # STEP 2 — Loss Calculation: measure prediction error

        loss = criterion(outputs['logits'], labels)


        # STEP 3 — Backward Pass: Compute gradients

        optimizer.zero_grad()

        loss.backward()


        # STEP 4 — Optimization: update model weights

        optimizer.step()

        total_loss += loss.item()

        # Update training accuracy statistics
        _, predicted = torch.max(outputs['logits'], 1) # Use outputs['logits']
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    train_accuracy = 100 * correct / total # Calculate training accuracy after each epoch

    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.2f}%")


========== EPOCH 1 ==========
Train Loss: 0.0230
Train Accuracy: 99.26%

========== EPOCH 2 ==========
Train Loss: 0.0173
Train Accuracy: 99.43%

========== EPOCH 3 ==========
Train Loss: 0.0134
Train Accuracy: 99.57%


In [25]:
# =====================================================
# EVALUATION
# =====================================================

model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():

    for batch in test_loader:

        images = batch[0].to(device) #  Meaning batch["image"]
        labels = batch[1].to(device) #  Meaning batch["label"]

        outputs = model(images)

        _, predicted = torch.max(outputs['logits'], 1) # Use outputs['logits']

        test_total += labels.size(0)

        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 99.21%


# Comparing Two Approaches

Both approaches achieved similar accuracy results. Therefore, the comparison focuses on usability, flexibility, and training workflow.

| Feature | Manual Training Loop | Trainer API |
|---|---|---|
| Training Speed | Faster for custom implementations, depending on optimization | Slightly slower due to additional built-in features and abstractions |
| Flexibility | Very high, easy to customize every training step | Moderate flexibility with predefined workflow |
| Code Complexity | Requires more code and manual implementation | Requires less code and easier to use |
| Deployment Speed | Slower development because more components must be implemented manually | Faster experimentation and deployment |
| Mixed Precision Support | Requires manual setup or libraries| Built-in support through TrainingArguments |
| Best Use Cases | Research, experimentation, custom algorithms | Quick fine-tuning, benchmarking, production workflows |

## Conclusion

The manual training loop provides deeper understanding of the internal deep learning workflow, especially the four core steps:

1. Forward Pass  
2. Loss Calculation  
3. Backward Pass  
4. Optimization  

On the other hand, the Trainer API simplifies the training process and is more suitable for rapid development and standard training pipelines.